# Spark'a Giriş

Spark, tıpkı alıştığımız gibi DataFrame'lerle çalışmamıza izin verir. Buradaki önemli konu, **Spark'ın büyük veriler üzerinde çalışmak üzere tasarlandığı** gerçeğinde ortaya çıkıyor. Bu çalışmada Spark kullanarak neler yapabildiğimizi keşfermeye çalışacağız.

>**HATIRLATMA:** Spark'ı kendi bilgisayarımızda kullandığımızda çok ciddi performans farkları göremeyebiliriz. Ama Spark'ın en güçlü olduğu konulardan biri de zaten birden çok makinede aynı anda çalışabilmesi :)

Bu dersin sonunda şunları yapabiliyor olmayı hedefliyoruz:

1. PySpark'ın syntaxını tanımak
2. PySpark kullanarak çeşitli sorgulama ve filtreleme işlemleri yapmak
3. SparkSQL'den yararlanmak

In [0]:
import pyspark

spark = pyspark.sql.SparkSession.builder.getOrCreate() # PySparkı tanımlama

In [0]:
cars = spark.read.csv('dbfs:/FileStore/tables/auto_mpg.csv', header='True', inferSchema='True')

In [0]:
type(cars)

Out[3]: pyspark.sql.dataframe.DataFrame

Bakalım nasıl görünüyormuş?

In [0]:
cars

Out[4]: DataFrame[mpg: double, cylinders: int, displacement: double, horsepower: double, weight: int, acceleration: double, model_year: int, origin: int, car_name: string]

In [0]:
cars.show() # PySpark DataFrame

+----+---------+------------+----------+------+------------+----------+------+--------------------+
| mpg|cylinders|displacement|horsepower|weight|acceleration|model_year|origin|            car_name|
+----+---------+------------+----------+------+------------+----------+------+--------------------+
|18.0|        8|       307.0|     130.0|  3504|        12.0|        70|     1|chevrolet,chevell...|
|15.0|        8|       350.0|     165.0|  3693|        11.5|        70|     1|   buick,skylark,320|
|18.0|        8|       318.0|     150.0|  3436|        11.0|        70|     1|  plymouth,satellite|
|16.0|        8|       304.0|     150.0|  3433|        12.0|        70|     1|       amc,rebel,sst|
|17.0|        8|       302.0|     140.0|  3449|        10.5|        70|     1|         ford,torino|
|15.0|        8|       429.0|     198.0|  4341|        10.0|        70|     1|    ford,galaxie,500|
|14.0|        8|       454.0|     220.0|  4354|         9.0|        70|     1|    chevrolet,impala|


In [0]:
cars.show(5)

+----+---------+------------+----------+------+------------+----------+------+--------------------+
| mpg|cylinders|displacement|horsepower|weight|acceleration|model_year|origin|            car_name|
+----+---------+------------+----------+------+------------+----------+------+--------------------+
|18.0|        8|       307.0|     130.0|  3504|        12.0|        70|     1|chevrolet,chevell...|
|15.0|        8|       350.0|     165.0|  3693|        11.5|        70|     1|   buick,skylark,320|
|18.0|        8|       318.0|     150.0|  3436|        11.0|        70|     1|  plymouth,satellite|
|16.0|        8|       304.0|     150.0|  3433|        12.0|        70|     1|       amc,rebel,sst|
|17.0|        8|       302.0|     140.0|  3449|        10.5|        70|     1|         ford,torino|
+----+---------+------------+----------+------+------------+----------+------+--------------------+
only showing top 5 rows



In [0]:
cars.printSchema() # Pandastaki info() fonksiyonuyla benzer

root
 |-- mpg: double (nullable = true)
 |-- cylinders: integer (nullable = true)
 |-- displacement: double (nullable = true)
 |-- horsepower: double (nullable = true)
 |-- weight: integer (nullable = true)
 |-- acceleration: double (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- origin: integer (nullable = true)
 |-- car_name: string (nullable = true)



In [0]:
cars.describe().show()

+-------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+
|summary|              mpg|         cylinders|      displacement|        horsepower|            weight|      acceleration|       model_year|            origin|            car_name|
+-------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+
|  count|              392|               392|               392|               392|               392|               392|              392|               392|                 392|
|   mean|23.44591836734694| 5.471938775510204|194.41198979591837|104.46938775510205|2977.5841836734694|15.541326530612228| 75.9795918367347|1.5765306122448979|                null|
| stddev|7.805007486571802|1.7057832474527845|104.64400390890465| 38.49115993282846| 849.402560

## Filtreleme İşlemleri

In [0]:
cars.filter(cars.model_year >75).show(5)

+----+---------+------------+----------+------+------------+----------+------+------------+
| mpg|cylinders|displacement|horsepower|weight|acceleration|model_year|origin|    car_name|
+----+---------+------------+----------+------+------------+----------+------+------------+
|28.0|        4|       107.0|      86.0|  2464|        15.5|        76|     2|    fiat,131|
|25.0|        4|       116.0|      81.0|  2220|        16.9|        76|     2|   opel,1900|
|25.0|        4|       140.0|      92.0|  2572|        14.9|        76|     1|    capri,ii|
|26.0|        4|        98.0|      79.0|  2255|        17.7|        76|     1|  dodge,colt|
|27.0|        4|       101.0|      83.0|  2202|        15.3|        76|     2|renault,12tl|
+----+---------+------------+----------+------+------------+----------+------+------------+
only showing top 5 rows



Aynı pandasta yaptığımız gibi tabiki birden çok filtre de uygulayabiliriz.

In [0]:
cars.filter(cars.model_year >75).filter(cars.weight > 2300).show(5)

+----+---------+------------+----------+------+------------+----------+------+--------------------+
| mpg|cylinders|displacement|horsepower|weight|acceleration|model_year|origin|            car_name|
+----+---------+------------+----------+------+------------+----------+------+--------------------+
|28.0|        4|       107.0|      86.0|  2464|        15.5|        76|     2|            fiat,131|
|25.0|        4|       140.0|      92.0|  2572|        14.9|        76|     1|            capri,ii|
|17.5|        8|       305.0|     140.0|  4215|        13.0|        76|     1|chevrolet,chevell...|
|16.0|        8|       318.0|     150.0|  4190|        13.0|        76|     1|dodge,coronet,bro...|
|15.5|        8|       304.0|     120.0|  3962|        13.9|        76|     1|         amc,matador|
+----+---------+------------+----------+------+------------+----------+------+--------------------+
only showing top 5 rows



In [0]:
cars.filter((cars.model_year >75) & (cars.weight > 2300)).show(5)

+----+---------+------------+----------+------+------------+----------+------+--------------------+
| mpg|cylinders|displacement|horsepower|weight|acceleration|model_year|origin|            car_name|
+----+---------+------------+----------+------+------------+----------+------+--------------------+
|28.0|        4|       107.0|      86.0|  2464|        15.5|        76|     2|            fiat,131|
|25.0|        4|       140.0|      92.0|  2572|        14.9|        76|     1|            capri,ii|
|17.5|        8|       305.0|     140.0|  4215|        13.0|        76|     1|chevrolet,chevell...|
|16.0|        8|       318.0|     150.0|  4190|        13.0|        76|     1|dodge,coronet,bro...|
|15.5|        8|       304.0|     120.0|  3962|        13.9|        76|     1|         amc,matador|
+----+---------+------------+----------+------+------------+----------+------+--------------------+
only showing top 5 rows



In [0]:
cars.select(['displacement','car_name']).show(5)

+------------+--------------------+
|displacement|            car_name|
+------------+--------------------+
|       307.0|chevrolet,chevell...|
|       350.0|   buick,skylark,320|
|       318.0|  plymouth,satellite|
|       304.0|       amc,rebel,sst|
|       302.0|         ford,torino|
+------------+--------------------+
only showing top 5 rows



In [0]:
cars.filter(cars.model_year >75).filter(cars.weight > 2300).select(['weight','car_name']).show(5)

+------+--------------------+
|weight|            car_name|
+------+--------------------+
|  2464|            fiat,131|
|  2572|            capri,ii|
|  4215|chevrolet,chevell...|
|  4190|dodge,coronet,bro...|
|  3962|         amc,matador|
+------+--------------------+
only showing top 5 rows



## Sütunlar Üzerinde İşlemler

In [0]:
cars.withColumn('new_column', cars.car_name).show(5) # Car Name sütunundaki verileri kullanarak yeni sütun oluştur

+----+---------+------------+----------+------+------------+----------+------+--------------------+--------------------+
| mpg|cylinders|displacement|horsepower|weight|acceleration|model_year|origin|            car_name|          new_column|
+----+---------+------------+----------+------+------------+----------+------+--------------------+--------------------+
|18.0|        8|       307.0|     130.0|  3504|        12.0|        70|     1|chevrolet,chevell...|chevrolet,chevell...|
|15.0|        8|       350.0|     165.0|  3693|        11.5|        70|     1|   buick,skylark,320|   buick,skylark,320|
|18.0|        8|       318.0|     150.0|  3436|        11.0|        70|     1|  plymouth,satellite|  plymouth,satellite|
|16.0|        8|       304.0|     150.0|  3433|        12.0|        70|     1|       amc,rebel,sst|       amc,rebel,sst|
|17.0|        8|       302.0|     140.0|  3449|        10.5|        70|     1|         ford,torino|         ford,torino|
+----+---------+------------+---

In [0]:
cars.withColumn('new_column', cars.car_name).select(['car_name','new_column']).show(5)

+--------------------+--------------------+
|            car_name|          new_column|
+--------------------+--------------------+
|chevrolet,chevell...|chevrolet,chevell...|
|   buick,skylark,320|   buick,skylark,320|
|  plymouth,satellite|  plymouth,satellite|
|       amc,rebel,sst|       amc,rebel,sst|
|         ford,torino|         ford,torino|
+--------------------+--------------------+
only showing top 5 rows



In [0]:
cars.withColumn('year_old', cars.model_year + 10).select(['model_year','year_old']).show(5)

+----------+--------+
|model_year|year_old|
+----------+--------+
|        70|      80|
|        70|      80|
|        70|      80|
|        70|      80|
|        70|      80|
+----------+--------+
only showing top 5 rows



In [0]:
cars.withColumns({'year_old': cars.model_year - 5, 'year_new': cars.model_year + 5}).select(['year_old', 'model_year','year_new']).show(5)

+--------+----------+--------+
|year_old|model_year|year_new|
+--------+----------+--------+
|      65|        70|      75|
|      65|        70|      75|
|      65|        70|      75|
|      65|        70|      75|
|      65|        70|      75|
+--------+----------+--------+
only showing top 5 rows



## GroupBy

In [0]:
cars.groupby('model_year').mean('acceleration').orderBy('model_year').show()

+----------+------------------+
|model_year| avg(acceleration)|
+----------+------------------+
|        70|12.948275862068966|
|        71|              15.0|
|        72|            15.125|
|        73|           14.3125|
|        74|16.173076923076923|
|        75|             16.05|
|        76|15.941176470588232|
|        77|15.435714285714285|
|        78|15.805555555555552|
|        79|15.813793103448274|
|        80| 17.01851851851852|
|        81|16.325000000000006|
|        82|             16.51|
+----------+------------------+



In [0]:
cars.groupby('model_year').count().orderBy('count', ascending=False).show()

+----------+-----+
|model_year|count|
+----------+-----+
|        73|   40|
|        78|   36|
|        76|   34|
|        82|   30|
|        75|   30|
|        70|   29|
|        79|   29|
|        81|   28|
|        72|   28|
|        77|   28|
|        80|   27|
|        71|   27|
|        74|   26|
+----------+-----+



## User Defined Functions (UDF)

In [0]:
def squared(number):
    return number ** 2

# SparkSQL'de kullanabileceğimiz fonksiyonu kaydediyoruz
spark.udf.register("squaredWithPython", squared)

Out[20]: <function __main__.squared(number)>

In [0]:
from pyspark.sql.functions import udf

squared_udf = udf(lambda x: squared(x)) # UDF kullanıma hazır

cars.select("mpg", squared_udf('mpg').alias("mpg_squared")).show()

+----+-----------+
| mpg|mpg_squared|
+----+-----------+
|18.0|      324.0|
|15.0|      225.0|
|18.0|      324.0|
|16.0|      256.0|
|17.0|      289.0|
|15.0|      225.0|
|14.0|      196.0|
|14.0|      196.0|
|14.0|      196.0|
|15.0|      225.0|
|15.0|      225.0|
|14.0|      196.0|
|15.0|      225.0|
|14.0|      196.0|
|24.0|      576.0|
|22.0|      484.0|
|18.0|      324.0|
|21.0|      441.0|
|27.0|      729.0|
|26.0|      676.0|
+----+-----------+
only showing top 20 rows



## SparkSQL

Geldik Spark'ın en güzel özelliklerden birine! SQL dilinin sorgulama işlemlerinde çok güçlü bir araç olduğunu biliyoruz. O zaman neden SQL kullanmıyoruz?

In [0]:
cars.createOrReplaceTempView("cars") # Kullanılabilir ve değiştirilebilir TempView oluşturma

Geriye sadece SQL yazmak kaldı :)

In [0]:
spark.sql('SELECT car_name, mpg, model_year FROM cars WHERE model_year=75').show()

+--------------------+----+----------+
|            car_name| mpg|model_year|
+--------------------+----+----------+
|plymouth,valiant,...|19.0|        75|
|      chevrolet,nova|18.0|        75|
|     mercury,monarch|15.0|        75|
|       ford,maverick|15.0|        75|
|    pontiac,catalina|16.0|        75|
|   chevrolet,bel,air|15.0|        75|
| plymouth,grand,fury|16.0|        75|
|            ford,ltd|14.0|        75|
|       buick,century|17.0|        75|
|chevroelt,chevell...|16.0|        75|
|         amc,matador|15.0|        75|
|       plymouth,fury|18.0|        75|
|       buick,skyhawk|21.0|        75|
| chevrolet,monza,2+2|20.0|        75|
|     ford,mustang,ii|13.0|        75|
|      toyota,corolla|29.0|        75|
|          ford,pinto|23.0|        75|
|         amc,gremlin|20.0|        75|
|       pontiac,astro|23.0|        75|
|       toyota,corona|24.0|        75|
+--------------------+----+----------+
only showing top 20 rows



In [0]:
type(spark.sql('SELECT car_name, mpg, model_year FROM cars WHERE model_year=75'))

Out[24]: pyspark.sql.dataframe.DataFrame

Veri setimizi ilk okuttuğumuzda oluşan DataFrame'in tipini hatırlıyor musunuz?

Demek ki SQL sorgularını kullanarak da DataFrameler üzerinden istediğimiz işlemleri gerçekleştirebiliyoruz.

In [0]:
cmd = '''SELECT MAX(mpg), model_year 
FROM cars 
GROUP BY model_year'''

spark.sql(cmd).show()

+--------+----------+
|max(mpg)|model_year|
+--------+----------+
|    43.1|        78|
|    39.1|        81|
|    33.0|        76|
|    28.0|        72|
|    36.0|        77|
|    44.0|        82|
|    46.6|        80|
|    29.0|        73|
|    27.0|        70|
|    33.0|        75|
|    35.0|        71|
|    37.3|        79|
|    32.0|        74|
+--------+----------+



In [0]:
cars.filter(cars.model_year >= 75).createOrReplaceTempView("cars") # Oluşturduğumuz View güncellendi

In [0]:
cmd = '''SELECT MAX(mpg), model_year
FROM cars
GROUP BY model_year 
ORDER BY model_year'''

spark.sql(cmd).show()

+--------+----------+
|max(mpg)|model_year|
+--------+----------+
|    33.0|        75|
|    33.0|        76|
|    36.0|        77|
|    43.1|        78|
|    37.3|        79|
|    46.6|        80|
|    39.1|        81|
|    44.0|        82|
+--------+----------+



Yukarıda oluşturduğumuz UDF da kullanalım :)

In [0]:
cmd = '''SELECT MAX(mpg), model_year, squaredWithPython(MAX(mpg))
FROM cars 
GROUP BY model_year'''

spark.sql(cmd).show()

+--------+----------+---------------------------+
|max(mpg)|model_year|squaredWithPython(max(mpg))|
+--------+----------+---------------------------+
|    43.1|        78|         1857.6100000000001|
|    39.1|        81|         1528.8100000000002|
|    33.0|        76|                     1089.0|
|    36.0|        77|                     1296.0|
|    44.0|        82|                     1936.0|
|    46.6|        80|                    2171.56|
|    33.0|        75|                     1089.0|
|    37.3|        79|         1391.2899999999997|
+--------+----------+---------------------------+



## Veri Setini Düzenleme

In [0]:
cars2 = spark.read.csv('dbfs:/FileStore/tables/auto_mpg.csv', header='True')

In [0]:
cars2.printSchema()

root
 |-- mpg: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- displacement: string (nullable = true)
 |-- horsepower: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- acceleration: string (nullable = true)
 |-- model_year: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- car_name: string (nullable = true)



In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import *

cars3 = cars2.withColumn("mpg",col("mpg").cast(FloatType()))\
.withColumn("cylinders",col("cylinders").cast(IntegerType()))\
.withColumn("horsepower",col("horsepower").cast(FloatType()))\
.withColumn("displacement",col("displacement").cast(FloatType()))\
.withColumn("weight",col("weight").cast(IntegerType()))\
.withColumn("acceleration",col("acceleration").cast(FloatType()))\
.withColumn("model_year",col("model_year").cast(IntegerType()))\
.withColumn("origin",col("origin").cast(IntegerType()))\
.withColumn("car_name",col("car_name").cast(StringType()))

In [0]:
cars3.printSchema()

root
 |-- mpg: float (nullable = true)
 |-- cylinders: integer (nullable = true)
 |-- displacement: float (nullable = true)
 |-- horsepower: float (nullable = true)
 |-- weight: integer (nullable = true)
 |-- acceleration: float (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- origin: integer (nullable = true)
 |-- car_name: string (nullable = true)



## PySpark SQL Fonksiyonları

In [0]:
from pyspark.sql.functions import split, explode, lower, sort_array
(
cars.withColumn('split_names', split(lower(col('car_name')), ","))
    .select('split_names','car_name').show(10, truncate=False)
)

+-----------------------------+-------------------------+
|split_names                  |car_name                 |
+-----------------------------+-------------------------+
|[chevrolet, chevelle, malibu]|chevrolet,chevelle,malibu|
|[buick, skylark, 320]        |buick,skylark,320        |
|[plymouth, satellite]        |plymouth,satellite       |
|[amc, rebel, sst]            |amc,rebel,sst            |
|[ford, torino]               |ford,torino              |
|[ford, galaxie, 500]         |ford,galaxie,500         |
|[chevrolet, impala]          |chevrolet,impala         |
|[plymouth, fury, iii]        |plymouth,fury,iii        |
|[pontiac, catalina]          |pontiac,catalina         |
|[amc, ambassador, dpl]       |amc,ambassador,dpl       |
+-----------------------------+-------------------------+
only showing top 10 rows



In [0]:
(
cars.withColumn('split_names', explode(split(lower(col('car_name')), ",")))
    .select('split_names','car_name').show(10)
)

+-----------+--------------------+
|split_names|            car_name|
+-----------+--------------------+
|  chevrolet|chevrolet,chevell...|
|   chevelle|chevrolet,chevell...|
|     malibu|chevrolet,chevell...|
|      buick|   buick,skylark,320|
|    skylark|   buick,skylark,320|
|        320|   buick,skylark,320|
|   plymouth|  plymouth,satellite|
|  satellite|  plymouth,satellite|
|        amc|       amc,rebel,sst|
|      rebel|       amc,rebel,sst|
+-----------+--------------------+
only showing top 10 rows



In [0]:
(
cars.withColumn('split_names', sort_array(split(lower(col('car_name')), "")))
    .select('split_names','car_name').show(10)
)

+--------------------+--------------------+
|         split_names|            car_name|
+--------------------+--------------------+
|[, ,, ,, a, b, c,...|chevrolet,chevell...|
|[, ,, ,, 0, 2, 3,...|   buick,skylark,320|
|[, ,, a, e, e, h,...|  plymouth,satellite|
|[, ,, ,, a, b, c,...|       amc,rebel,sst|
|[, ,, d, f, i, n,...|         ford,torino|
|[, ,, ,, 0, 0, 5,...|    ford,galaxie,500|
|[, ,, a, a, c, e,...|    chevrolet,impala|
|[, ,, ,, f, h, i,...|   plymouth,fury,iii|
|[, ,, a, a, a, a,...|    pontiac,catalina|
|[, ,, ,, a, a, a,...|  amc,ambassador,dpl|
+--------------------+--------------------+
only showing top 10 rows



## Pandas ve PySpark Arasında Nasıl Karar Verilir? 

- Verileriniz çok büyükse ve yıllar içinde önemli ölçüde büyüyorsa ve işlem sürenizi iyileştirmek istiyorsanız.
- Hataya dayanıklı istiyorsanız. ANSI SQL uyumluluğu. Seçilecek dil (Spark Python, Scala, Java ve R'yi destekler)
- Makine öğrenimi yeteneği istediğinizde. Parke, Avro, Hive, Casandra, Snowflake e.t.c okumak istiyorum
- Verileri gerçek zamanlı olarak işlemek istiyorsanız.

> **Güzel Bir Kaynak:** https://sparkbyexamples.com/pyspark